In [ ]:
import os
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [2]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

vector_words_model = KeyedVectors.load_word2vec_format('./models/cc.ru.300.vec.gz', binary=False)

In [3]:
def get_vector(word):
    try:
        return vector_words_model[word]
    except KeyError:
        return None

def calculate_similarity(tag1, tag2):
    vec1 = get_vector(tag1)
    vec2 = get_vector(tag2)
    if vec1 is not None and vec2 is not None:
        return cosine_similarity([vec1], [vec2])[0][0]
    else:
        return 0.0

In [4]:
class Model:
    def __init__(self, mod_path, tok_path):
        self.model_path = mod_path
        self.tokenizer_path = tok_path

    def switch(self, other_model):
        self.model_path = other_model.model_path
        self.tokenizer_path = other_model.tokenizer_path
        self.load()

    def load(self):
        self.model = tf.keras.models.load_model(self.model_path)
        with open(self.tokenizer_path, 'r', encoding='utf-8') as f:
            tokenizer_json = f.read()
            self.tokenizer = tokenizer_from_json(tokenizer_json)
        self.vocab_size = len(self.tokenizer.word_index)+1


In [5]:
MODEL_28_09_24 = Model("./models/28_09_2024.keras", "./models/28_09_2024.json")
MODEL_30_09_24 = Model("./models/30_09_2024.keras", "./models/28_09_2024.json")
MODEL_04_10_24 = Model("./models/04_10_2024.keras", "./models/04_10_2024.json")
MODEL_06_10_24 = Model("./models/06_10_2024.keras", "./models/06_10_2024.json")
MODEL_08_10_24 = Model("./models/08_10_2024.keras", "./models/08_10_2024.json")

MAX_LENGTH = 32  # максимальное количество токенов на выходе

In [6]:
def find_image(dir, num):
    return next(filter(lambda f: f.startswith(f"{num}."), os.listdir(dir)))

def generate_desc(model, tokenizer, photo):
    sequence = [tokenizer.word_index['start']]

    sequence_input = np.array([sequence[-1]])
    result = np.argmax(model.predict([photo, sequence_input], verbose=0), axis=-1)
    return [tokenizer.index_word.get(token) for token in filter(None, result[0])]

In [17]:
current_model = Model("", "")
current_model.switch(MODEL_08_10_24)

In [ ]:
dir_path = "./images/"
for num in range(543, 544):
    # Загрузка изображения
    photo_path = dir_path+find_image(dir_path, num)
    photo = load_img(photo_path, target_size=(512, 512))
    photo = img_to_array(photo)
    photo = np.expand_dims(photo, axis=0)  # Добавляем размерность batch
    photo = preprocess_input(photo)

    description = generate_desc(current_model.model, current_model.tokenizer, photo)
    print(f"{num} ---", description)

In [9]:
with open("./train/descriptions.json", "r", encoding="utf-8") as file:
    real_descriptions = json.load(file)

In [15]:
class Mark:
    def __init__(self, real, predict):
        self.real = real
        self.predict = predict

        predicted_set = set(predict)
        real_set = set(real)

        self.common = real_set.intersection(predicted_set)
        self.extra = predicted_set.difference(real_set)
        self.unused = real_set.difference(predicted_set)

        try:
            self.extra_unused_similarity_grades=[]
            for e_tag in self.extra:
                self.extra_unused_similarity_grades.append((e_tag, max([calculate_similarity(e_tag, u_tag) for u_tag in self.unused])))
        except ValueError:
            pass
        
        try:
            self.extra_common_similarity_grades=[]
            for e_tag in self.extra:
                self.extra_common_similarity_grades.append((e_tag, max([calculate_similarity(e_tag, c_tag) for c_tag in self.common])))
        except ValueError:
            pass

    def summary(self):
       print("Количество совпавших тегов:", len(self.common))
       print("Количество добавленных тегов:", len(self.extra))
       print("Количество потерянных тегов:", len(self.unused))
       print("Насколько лишние теги подходят вместо неиспользованных:", self.extra_unused_similarity_grades)
       print("Насколько лишние теги подходят, как дополнения к совпавшим:", self.extra_common_similarity_grades)

    def mark(self):
        common_weight = 0.2
        extra_common_weight = 0.4
        extra_unused_weight = 0.4
        result=0
        add1=0
        add2=0
        result+=(len(self.common)/len(self.real))*common_weight
        try:
            result+=(sum(map(lambda pair: pair[1], self.extra_common_similarity_grades))/len(self.extra_common_similarity_grades))*extra_common_weight
        except:
            if result!=common_weight:
                add1=0*extra_common_weight
            else:
                add1=1*extra_common_weight
        try:
            result+=(sum(map(lambda pair: pair[1], self.extra_unused_similarity_grades))/len(self.extra_unused_similarity_grades))*extra_unused_weight
        except:
            if result!=common_weight:
                add2=0*extra_unused_weight
            else:
                add2=1*extra_unused_weight
        return result+add1+add2

In [ ]:
dir_path = "./train/"

def get_image_desc(number):
    photo_path = dir_path+find_image(dir_path, number)
    photo = load_img(photo_path, target_size=(512, 512))
    photo = img_to_array(photo)
    photo = np.expand_dims(photo, axis=0)  # Добавляем размерность batch
    photo = preprocess_input(photo)

    # Генерация описания
    return generate_desc(current_model.model, current_model.tokenizer, photo)

def get_model_mark(tests_count, verbose=False):
    global_mark = 0
    for i in range(1, tests_count+1):
        test_number = str(i)
        descriptions = get_image_desc(test_number)
        real = real_descriptions[test_number]
        mark = Mark(real, descriptions)
        res = mark.mark()
        if verbose:
            print(f"============ test {i} ============")
            print("hyp:", descriptions)
            print("ref:", real)
            mark.summary()
            print("res_mark:", res*100)
        global_mark+=res
    return global_mark/tests_count*100

get_model_mark(20, verbose=False)

